# Ombudsman decisions: Revolut, Monzo and Starling (2025)

Collects every published Financial Ombudsman decision for Revolut, Monzo and Starling
in the banking sector for 2025, parses each result, and classifies it into one complaint area.

Run order:

1. Fetch one page and inspect the HTML structure
2. Scrape all pages for the three firms, with caching
3. Data quality checks: unparsed rows and firm-name contamination
4. Classify each decision into a complaint area
5. Validate the classifier against two independent hand-checked samples

Findings, charts and limitations are in the README.

## 1. Fetch one page

The decisions search is paginated 10 results at a time via the `Start` parameter.
robots.txt permits access to this path. Requests are spaced one second apart and
every page is cached locally, so nothing is fetched twice.

In [ ]:
import requests

url = "https://www.financial-ombudsman.org.uk/businesses/resolving-complaint/ombudsman-decisions/search"
params = {
    "BusinessName": "Revolut",
    "Business": "Revolut",
    "IndustrySectorID[1]": "1",
    "DateFrom": "2025-01-01",
    "DateTo": "2025-12-31",
    "Sort": "date",
    "Start": "0",
}
headers = {"User-Agent": "student-research-project/1.0 (personal study)"}

r = requests.get(url, params=params, headers=headers, timeout=30)
print(r.status_code, len(r.text))
print(r.url)

Each result sits in its own `<li>` with no class name, so results are identified by content:
any list item containing a decision reference.

In [ ]:
import re
from bs4 import BeautifulSoup

soup = BeautifulSoup(r.text, "html.parser")

node = soup.find(string=re.compile("DRN-"))
print(repr(node)[:200])

parent = node.find_parent(["li", "article", "div"])
print(parent.name, parent.get("class"))
print(parent.get_text(" ", strip=True)[:400])

## 2. Parse and scrape

One result's flat text runs: reference, date, firm, outcome, product, then the complaint
summary. The summary sometimes repeats the reference and sometimes doesn't, so that part
is optional. `parse_item` returns `None` when the pattern doesn't fit, which lets the
scraper count failures instead of silently dropping rows.

In [ ]:
import os, time
import pandas as pd

PRODUCTS = ["Banking and Payments", "Consumer Credit", "Insurance",
            "Investments", "Mortgages and Home Finance"]

HEAD = re.compile(
    r"Decision Reference (DRN-\d+)\s+(\d{1,2} \w{3} \d{4})\s+(.+?)\s+"
    r"(Not upheld|Upheld)\s+(.*)", re.S)

def parse_item(li):
    m = HEAD.search(li.get_text(" ", strip=True))
    if not m:
        return None
    ref, date, firm, outcome, rest = m.groups()
    rest = rest.split("View decision")[0].strip()
    rest = re.sub(r"\(\d+ pages?\)\s*$", "", rest).strip()
    product = next((p for p in sorted(PRODUCTS, key=len, reverse=True)
                    if rest.startswith(p)), None)
    if product:
        rest = rest[len(product):].strip()
    rest = re.sub(r"^DRN-\d+\s*", "", rest)      # repeated reference, when present
    return {"ref": ref, "date": date, "firm": firm,
            "outcome": outcome, "product": product, "summary": rest}

A quick unit test for the parser, using a result that failed under an earlier pattern
because its summary did not repeat the decision reference.

In [ ]:
text = ("Decision Reference DRN-5247067 28 May 2025 Revolut Ltd Not upheld "
        "Banking and Payments The complaint Mrs A has complained about Revolut Ltd "
        "not refunding several payments she says she made and lost to a scam. "
        "What happened The background... (3 pages) View decision")

class Fake:                      # stand-in for a BeautifulSoup tag
    def get_text(self, *a, **k): return text

print(parse_item(Fake()))

In [ ]:
os.makedirs("pages", exist_ok=True)

def fetch_page(business, start):
    path = f"pages/{business}_{start}.html"
    if os.path.exists(path):                      # already downloaded: reuse
        return open(path, encoding="utf-8").read()
    p = dict(params, BusinessName=business, Business=business, Start=str(start))
    r = requests.get(url, params=p, headers=headers, timeout=30)
    r.raise_for_status()
    open(path, "w", encoding="utf-8").write(r.text)
    time.sleep(1)                                 # be polite
    return r.text

def parse_page(html):
    soup = BeautifulSoup(html, "html.parser")
    items = [li for li in soup.find_all("li")
             if li.find(string=re.compile(r"Decision Reference DRN-"))]
    return [parse_item(li) for li in items]

def scrape(business, max_pages=None):
    rows, start, pages = [], 0, 0
    while True:
        parsed = parse_page(fetch_page(business, start))
        if not parsed:
            break
        unparsed = sum(r is None for r in parsed)
        if unparsed:
            print(f"  {business} start={start}: {unparsed} rows failed to parse")
        rows += [dict(r, search_firm=business) for r in parsed if r]
        start += 10
        pages += 1
        if max_pages and pages >= max_pages:
            break
    print(f"{business}: {len(rows)} rows from {pages} pages")
    return rows

Test on three pages before running the full scrape.

In [ ]:
test = scrape("Revolut", max_pages=3)
len(test)          # expect 30

In [ ]:
all_rows = []
for business in ["Revolut", "Monzo", "Starling"]:
    all_rows += scrape(business)

df_raw = pd.DataFrame(all_rows).drop_duplicates("ref")
print(len(all_rows), "rows scraped;", len(df_raw), "after removing duplicate references")
df_raw.groupby(["search_firm", "outcome"]).size()

## 3. Data quality checks

Two problems, both found by checking rather than by assuming.

**Unparsed rows.** Any result that doesn't match the pattern is printed above. The cell
below pulls the raw text of each one from the cache so the cause can be identified.

**Firm-name contamination.** The site matches business names by substring, so a search for
"Starling" also returns *Startline Motor Finance*. Grouping by the firm named in each
decision against the firm searched for makes this visible.

In [ ]:
def failures(business):
    out = []
    for f in sorted(os.listdir("pages")):
        if not f.startswith(business + "_"):
            continue
        soup = BeautifulSoup(open(f"pages/{f}", encoding="utf-8").read(), "html.parser")
        for li in soup.find_all("li"):
            if li.find(string=re.compile(r"Decision Reference DRN-")) and parse_item(li) is None:
                out.append((f, li.get_text(" ", strip=True)[:300]))
    return out

for f, text in failures("Revolut") + failures("Monzo") + failures("Starling"):
    print(f, "|", text, "\n")

In [ ]:
df_raw.groupby(["search_firm", "firm"]).size()

In [ ]:
KEEP = df_raw["firm"].str.contains("Revolut|Monzo|Starling Bank", case=False)
print("dropping:", (~KEEP).sum())
print(df_raw.loc[~KEEP, "firm"].value_counts())

df = df_raw[KEEP].copy()
df.to_csv("fos_decisions_2025.csv", index=False)

print(len(df), "rows")
print(df.groupby(["search_firm", "outcome"]).size())
print("missing product:", df["product"].isna().sum())

Scrape reconciled against the totals published on the search page:
Revolut 1,727 of 1,727 and Starling exact; Monzo 943 of 944 (one decision missed, 0.1%).

## 4. Classify

Rules are applied in priority order, so every decision gets exactly one area and the
categories are mutually exclusive. The rules classify on **what went wrong, not what the
customer asked for**: a scam victim asking for a refund is a scam case, not a card dispute.

Two details that matter:

- Gambling requires both a gambling reference *and* a failure to protect or block, so a
  chargeback dispute about gambling deposits isn't miscounted.
- The source text uses curly apostrophes, so they are normalised before matching. Without
  this, `didn't authorise` never matches and unauthorised-payment cases are misfiled as scams.

In [ ]:
RULES = [
    ("Gambling and vulnerability",
        r"(gambling|self-exclu).{0,80}(protect|block|stop|prevent|vulnerab)"
        r"|(protect|block|stop|prevent).{0,80}(gambling|self-exclu)"),
    ("Fraud (unauthorised)", r"unauthorised|didn'?t (make|authorise)|did not authorise"),
    ("Scams",               r"scam|fell victim|impersonat|blackmail|extortion"),
    ("Cifas marker",        r"cifas|fraud marker"),
    ("Blocked accounts/payments",
        r"block|restrict|froze|frozen|clos(e|ed|ing)\b.{0,30}\baccount|account (clos|review)"),
    ("Card disputes",
        r"chargeback|merchant|disputed transaction|debit card|cash machine|atm"
        r"|goods|not received|overcharg|refund"),
]

def classify(summary):
    text = str(summary).lower().replace("\u2019", "'")
    for area, pattern in RULES:
        if re.search(pattern, text):
            return area
    return "Other"

df["area"] = df["summary"].apply(classify)
df.to_csv("fos_classified_2025.csv", index=False)

print(pd.crosstab(df["area"], df["search_firm"]))
print()
print(df.groupby(["search_firm", "area"])["outcome"]
        .apply(lambda s: round(100 * (s == "Upheld").mean(), 1)).unstack(0))

## 5. Validate the classifier

The rules were refined against the first sample of 30, then tested against a second,
independent sample of 30. Scoring on the sample used to build the rules would overstate
accuracy, so the second sample is the honest measure: **90% agreement** with manual
classification. The disagreements were mixed scam/unauthorised cases and payment types
outside the card rule, such as direct debits.

In [ ]:
check = df.sample(30, random_state=1)[["ref", "area", "summary"]]
for r in check.itertuples():
    print(r.area, "|", r.summary[:160], "\n")

In [ ]:
fresh = df.sample(30, random_state=7)[["ref", "area", "summary"]]
for r in fresh.itertuples():
    print(r.area, "|", r.summary[:160], "\n")

## 6. Export

`fos_classified_2025.csv` is the table used for the Power BI report and the README figures.

In [ ]:
from google.colab import files
files.download("fos_classified_2025.csv")